# Corpus Stats — Stage 0 EDA

Five questions, each tied to a downstream decision (see CHANGELOG 2026-07-05):

1. **Field coverage by era** → what can Stage 6 rely on, where do eras break
2. **Speaker-label picture** → is speaker-attribution post-processing worth it; reach of Stage 5 speaker filters
3. **Size distributions** → Stage 1 chunk size, embedding cost
4. **Flagged inventory** → the backfill plan
5. **Crypto series integration check** → (series, number) identity works

Reads the Manifest and Episode Records only; no network, no model calls.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("../data")
manifest = json.loads((DATA / "manifest.json").read_text())

rows, turn_tokens = [], []
for slug, e in manifest.items():
    if e["status"] != "parsed":
        continue
    r = json.loads((DATA / "raw" / f"{slug}.json").read_text())
    turns = r["transcript"]
    chars = sum(len(t["text"]) for t in turns)
    turn_tokens.extend(len(t["text"]) // 4 for t in turns)
    rows.append({
        "slug": slug,
        "series": r["series"],
        "episode": r["episode"],
        "date": r["date"],
        "n_turns": len(turns),
        "n_speakers": len(r["speakers"]),
        "unattributed": sum(1 for t in turns if not t["speaker"]),
        "unattributed_chars": sum(len(t["text"]) for t in turns if not t["speaker"]),
        "transcript_chars": chars,
        "tokens_est": chars // 4,  # crude chars/4 heuristic; real count arrives in Stage 1
        "has_summary": r["summary"] is not None,
        "n_key_points": len(r["key_points"]),
        "n_links": len(r["links"]),
    })
df = pd.DataFrame(rows).sort_values(["series", "episode"]).reset_index(drop=True)
df["date"] = pd.to_datetime(df["date"])
print(len(df), "Episode Records:", df["series"].value_counts().to_dict())

## Q1 — Optional-field coverage by era (main series)

In [ ]:
main = df[df["series"] == "main"].sort_values("episode")
cov = pd.DataFrame({
    "speaker labels": (main["n_speakers"] > 0).values,
    "summary": main["has_summary"].values,
    "key points": (main["n_key_points"] > 0).values,
    "links": (main["n_links"] > 0).values,
}, index=main["episode"]).rolling(25, min_periods=5).mean()
ax = cov.plot(figsize=(10, 4), title="Field coverage, rolling 25-episode share (main series)")
ax.set_xlabel("episode"); ax.set_ylabel("share with field"); ax.set_ylim(-0.05, 1.05)
plt.show()

overall = pd.Series({
    "speaker labels": (df["n_speakers"] > 0).mean(),
    "summary": df["has_summary"].mean(),
    "key points": (df["n_key_points"] > 0).mean(),
    "links": (df["n_links"] > 0).mean(),
}).mul(100).round(1).rename("coverage %")
print(overall.to_string())

## Q2 — Speaker-label picture

In [ ]:
no_labels = df[df["n_speakers"] == 0]
print(f"{len(no_labels)} records with no speaker labels "
      f"({100 * len(no_labels) / len(df):.0f}% of corpus), by series: "
      f"{no_labels['series'].value_counts().to_dict()}")
mn = no_labels[no_labels["series"] == "main"]["episode"]
print(f"main-series no-label episodes: {len(mn)}, range {mn.min()}-{mn.max()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(mn, bins=42)
axes[0].set_title("No-speaker-label episodes"); axes[0].set_xlabel("episode")
mm = df[df["series"] == "main"].sort_values("episode")
share = (mm["unattributed_chars"] / mm["transcript_chars"]).rolling(15, min_periods=3).mean()
axes[1].plot(mm["episode"], share)
axes[1].set_title("Unattributed share of transcript text (rolling 15)")
axes[1].set_xlabel("episode"); axes[1].set_ylim(-0.05, 1.05)
plt.tight_layout(); plt.show()

total_unattr = df["unattributed_chars"].sum() / df["transcript_chars"].sum()
print(f"corpus-wide, {100 * total_unattr:.1f}% of transcript text is unattributed")

## Q3 — Size distributions and chunk estimate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(df["tokens_est"] / 1000, bins=40)
axes[0].set_title("Transcript tokens per episode (est., k)")
axes[1].hist(df["n_turns"], bins=40)
axes[1].set_title("Speaker Turns per episode")
axes[2].hist([t for t in turn_tokens if t < 1500], bins=50)
axes[2].set_title("Tokens per Speaker Turn (est., <1500)")
plt.tight_layout(); plt.show()

total = df["tokens_est"].sum()
CHUNK, OVERLAP = 800, 0.15
print(f"corpus ≈ {total / 1e6:.1f}M transcript tokens (chars/4 heuristic)")
print(f"at {CHUNK}-token chunks, {int(OVERLAP * 100)}% overlap ≈ "
      f"{int(total / (CHUNK * (1 - OVERLAP))):,} chunks")
print(f"median episode: {df['tokens_est'].median():,.0f} tokens, "
      f"{df['n_turns'].median():.0f} turns; "
      f"median turn: {pd.Series(turn_tokens).median():.0f} tokens")

## Q4 — Flagged inventory (the backfill plan)

In [ ]:
from bs4 import BeautifulSoup

inv = []
for slug, e in manifest.items():
    if e["status"] != "flagged":
        continue
    soup = BeautifulSoup((DATA / "html" / f"{slug}.html").read_text(), "lxml")
    body = soup.select_one("article.entry")
    h1 = (body.find("h1") if body else None) or soup.find("h1")
    inv.append({
        "slug": slug,
        "title": h1.get_text(" ", strip=True)[:55] if h1 else None,
        "body_chars": len(body.get_text(" ", strip=True)) if body else 0,
        "flags": "; ".join(e["flags"]),
    })
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(inv).sort_values("slug").reset_index(drop=True)

## Q5 — Crypto series integration check

In [ ]:
crypto = df[df["series"] == "crypto"].sort_values("episode")
assert len(crypto) == 17, f"expected 17 crypto episodes, got {len(crypto)}"
assert crypto["episode"].tolist() == list(range(1, 18)), "crypto numbering not 1..17"
print("crypto series: 17/17 in the Corpus, numbered 1-17 from titles")
crypto[["episode", "n_turns", "n_speakers", "n_key_points", "n_links", "tokens_est"]]

## Second pass — speaker inventory & canonicalization

Entity resolution over ~300 labels via the curated table in
`scraper/speakers.py` (variants merged, junk dropped). Observed `speaker`
fields in records are never rewritten — canonicalization applies here and at
chunk time.

In [ ]:
from collections import Counter
from scraper.speakers import canonicalize, CANONICAL, JUNK, KNOWN_HOSTS

raw_c, canon_c = Counter(), Counter()
for slug, e in manifest.items():
    if e["status"] != "parsed":
        continue
    r = json.loads((DATA / "raw" / f"{slug}.json").read_text())
    for t in r["transcript"]:
        if t["speaker"]:
            raw_c[t["speaker"]] += 1
            if c := canonicalize(t["speaker"]):
                canon_c[c] += 1
merged = sum(raw_c[v] for v in CANONICAL)
print(f"{len(raw_c)} raw labels -> {len(canon_c)} canonical speakers "
      f"({merged} turns re-labeled by variant merge, {sum(raw_c[j] for j in JUNK if j in raw_c)} junk turns dropped)")
print(); print("top 10 canonical speakers by turns:")
for name, n in canon_c.most_common(10):
    host = " (host)" if name in KNOWN_HOSTS else ""
    print(f"  {n:6d}  {name}{host}")

## Second pass — attribution coverage after enrichment

Every turn lands in exactly one bucket. `observed` = the page printed a
label; `inferred` = answer-style turn attributed to the title's guest
(question turns deliberately stay unattributed — no in-episode evidence of
which hosts were present in 101/143 interview episodes).

In [ ]:
buckets = Counter()
chars = Counter()
for slug, e in manifest.items():
    if e["status"] != "parsed":
        continue
    r = json.loads((DATA / "raw" / f"{slug}.json").read_text())
    for t in r["transcript"]:
        if t["speaker"] and canonicalize(t["speaker"]):
            b = "observed"
        elif t.get("inferred_speaker"):
            b = "inferred (guest)"
        elif t.get("style") == "question":
            b = "unattributed question"
        elif t.get("style") == "boilerplate":
            b = "boilerplate"
        else:
            b = "unattributed other"
        buckets[b] += 1
        chars[b] += len(t["text"])
total = sum(chars.values())
print(f"{'bucket':24s} {'turns':>7s} {'share of text':>14s}")
for b, n in buckets.most_common():
    print(f"{b:24s} {n:7d} {100 * chars[b] / total:13.1f}%")

## Second pass — boilerplate scan

Repeated intro/outro/disclaimer text. Kept in records (they are what the
page says); chunking will tag it as metadata so retrieval can exclude it
without losing the page-faithful text.

In [ ]:
openings = Counter()
tail_hits = 0
boiler_chars = 0
total_chars = 0
for slug, e in manifest.items():
    if e["status"] != "parsed":
        continue
    r = json.loads((DATA / "raw" / f"{slug}.json").read_text())
    ts = r["transcript"]
    if ts:
        openings[ts[0]["text"][:55].lower()] += 1
    for t in ts:
        total_chars += len(t["text"])
        if t.get("style") == "boilerplate":
            boiler_chars += len(t["text"])
    last = " ".join(t["text"] for t in ts[-2:]).lower()
    if "error in the transcript" in last or "disclaimer" in last or "disclosure" in last:
        tail_hits += 1
n_eps = sum(1 for e in manifest.values() if e["status"] == "parsed")
print("most common transcript openings (casefolded):")
for txt, n in openings.most_common(3):
    print(f"  {n:4d}x {txt!r}")
print(f"episodes with disclaimer-ish tail text: {tail_hits}/{n_eps}")
print(f"boilerplate-style turns: {100 * boiler_chars / total_chars:.2f}% of transcript text")

## Findings

Numbers above regenerate on every run. Decision-relevant findings are
recorded in `CHANGELOG.md`; concepts in `docs/LEARNING.md`.